<a href="https://colab.research.google.com/github/Sakshiparve695/Smart-Delivery-Analytics-GCP/blob/main/Pyspark_Etl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, to_date, avg

# ---------------- CREATE SPARK SESSION ----------------
spark = SparkSession.builder \
    .appName("SmartDeliveryETL") \
    .getOrCreate()

# ---------------- SAMPLE DATA (SIMULATING MYSQL TABLE) ----------------
data = [
    (1, 101, "2024-01-01 10:00:00", "2024-01-01 10:40:00", 10),
    (2, 102, "2024-01-01 11:00:00", "2024-01-01 11:20:00", 5),
    (3, 101, "2024-01-01 12:00:00", "2024-01-01 12:50:00", 12),
    (4, 103, "2024-01-01 13:00:00", "2024-01-01 13:25:00", 7)
]

columns = ["delivery_id", "agent_id", "order_time", "delivery_time", "distance"]

df = spark.createDataFrame(data, columns)

# ---------------- CONVERT TO TIMESTAMP ----------------
df = df.withColumn("order_time", col("order_time").cast("timestamp")) \
       .withColumn("delivery_time", col("delivery_time").cast("timestamp"))

# ---------------- TRANSFORMATION ----------------

# Calculate delivery duration (in minutes)
df = df.withColumn(
    "delivery_duration",
    (unix_timestamp("delivery_time") - unix_timestamp("order_time")) / 60
)

# Delay logic
df = df.withColumn(
    "is_delayed",
    col("delivery_duration") > 30
)

# Extract delivery date
df = df.withColumn(
    "delivery_date",
    to_date("delivery_time")
)

# ---------------- FINAL FACT TABLE STRUCTURE ----------------
fact_df = df.select(
    "agent_id",
    "delivery_duration",
    "distance",
    "is_delayed",
    "delivery_date"
)

# ---------------- SHOW RESULT ----------------
print("=== Transformed Data (Fact Table) ===")
fact_df.show()

# ---------------- ANALYTICS ----------------

# Average delivery time per agent
print("=== Average Delivery Time per Agent ===")
fact_df.groupBy("agent_id").agg(
    avg("delivery_duration").alias("avg_delivery_time")
).show()

# Delay percentage
print("=== Delay Percentage ===")
total_count = fact_df.count()
delayed_count = fact_df.filter(col("is_delayed") == True).count()

delay_percentage = (delayed_count / total_count) * 100 if total_count > 0 else 0

print(f"Total Deliveries: {total_count}")
print(f"Delayed Deliveries: {delayed_count}")
print(f"Delay Percentage: {round(delay_percentage, 2)}%")

# ---------------- STOP SPARK SESSION ----------------
spark.stop()